In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load Data
df = pd.read_csv('/content/dataset_transaksi.csv')

# 2. Preprocessing Label
encoder = LabelEncoder()
y = encoder.fit_transform(df['kategori'])
num_classes = len(encoder.classes_)

# 3. Tokenisasi Teks
max_words = 5000
max_len = 20
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['deskripsi'])

X = tokenizer.texts_to_sequences(df['deskripsi'])
X = pad_sequences(X, maxlen=max_len, padding='post')

# 4. Split Data 8:2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Membangun Arsitektur Deep Learning (LSTM)
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(max_words, 64, input_shape=(max_len,)),
    tf.keras.layers.SpatialDropout1D(0.2),
    tf.keras.layers.LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 6. Training Model
print("Memulai pelatihan model...")
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), batch_size=32, verbose=1)

# 7. Simpan Model untuk Hugging Face
model.save('transaction_classifier_model.keras')
print("Model berhasil disimpan sebagai 'transaction_classifier_model.keras'")

Memulai pelatihan model yang diperbaiki...
Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - accuracy: 0.7879 - loss: 0.6833 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.9999 - loss: 0.0080 - val_accuracy: 1.0000 - val_loss: 1.7305e-04
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.9999 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 5.8290e-05
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.9999 - loss: 0.0018 - val_accuracy: 1.0000 - val_loss: 2.2051e-05
Epoch 5/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 1.0000 - loss: 0.0012 - val_accuracy: 1.0000 - val_loss: 1.0043e-05
Epoch 6/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 1.0000 - loss: 6.8857e-04 - val_accuracy: 1.0000 - val_loss: 6.3179e-06
Epoch 7/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 1.0000 - loss: 4.6206e-04 - val_accuracy: 1.0000 - val_loss: 3.4434e-06
Epoch 8/15
250/250 ━

### 2. Fungsi Prediksi

Gunakan kode di bawah ini untuk menguji model dengan teks deskripsi baru.

In [14]:
def predict_transaction(text):
    # Tokenize and Pad
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    # Predict
    prediction = model.predict(padded, verbose=0)
    class_idx = np.argmax(prediction, axis=1)[0]

    # Get Label
    category = encoder.inverse_transform([class_idx])[0]
    confidence = np.max(prediction)

    return category, confidence

# Contoh Penggunaan
contoh_teks = "beli baju"
kategori, skor = predict_transaction(contoh_teks)
print(f"Deskripsi: {contoh_teks}")
print(f"Prediksi Kategori: {kategori} (Confidence: {skor:.2f})")

Deskripsi: beli baju
Prediksi Kategori: Belanja (Confidence: 0.82)


### Download Model
Gunankan kode di bawah ini untuk mendownload file model `.keras` ke perangkat lokal Anda.

In [15]:
from google.colab import files

# Mendownload model yang telah disimpan
files.download('transaction_classifier_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 5. Menyimpan Aset Pendukung (Tokenizer & Label Encoder)

Model tidak bisa bekerja sendiri tanpa Tokenizer dan Label Encoder yang sama dengan saat pelatihan. Kita akan menyimpannya menggunakan `pickle`.

In [17]:
import pickle
import os
from google.colab import files

# Nama file yang dibutuhkan
model_file = 'transaction_classifier_model.keras'
tokenizer_file = 'tokenizer.pkl'
encoder_file = 'label_encoder.pkl'

# 1. Simpan Tokenizer dan Label Encoder dari memori ke disk
try:
    if 'tokenizer' in globals():
        with open(tokenizer_file, 'wb') as handle:
            pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
        print("✅ Tokenizer berhasil disimpan.")

    if 'encoder' in globals():
        with open(encoder_file, 'wb') as handle:
            pickle.dump(encoder, handle, protocol=pickle.HIGHEST_PROTOCOL)
        print("✅ Label Encoder berhasil disimpan.")
except Exception as e:
    print(f"⚠️ Gagal menyimpan aset: {e}")

# 2. Verifikasi keberadaan semua file sebelum di-zip
files_to_zip = [model_file, tokenizer_file, encoder_file]
missing_files = [f for f in files_to_zip if not os.path.exists(f)]

if not missing_files:
    # 3. Proses Zipping
    print("\n📦 Membuat paket model_assets.zip...")
    !zip -j model_assets.zip {' '.join(files_to_zip)}

    # 4. Download
    if os.path.exists('model_assets.zip'):
        print("🚀 Memulai unduhan...")
        files.download('model_assets.zip')
    else:
        print("❌ Gagal membuat file ZIP.")
else:
    print(f"\n❌ File berikut tidak ditemukan: {missing_files}")
    print("\n👉 Jika file tidak ditemukan, pastikan Anda telah menjalankan Sel Pelatihan (Sel #2) di atas agar variabel tersedia di memori.")

✅ Tokenizer berhasil disimpan.
✅ Label Encoder berhasil disimpan.

📦 Membuat paket model_assets.zip...
updating: transaction_classifier_model.keras (deflated 59%)
updating: tokenizer.pkl (deflated 34%)
updating: label_encoder.pkl (deflated 19%)
🚀 Memulai unduhan...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import textwrap

# 1. Update requirements.txt untuk Gradio
with open('requirements.txt', 'w') as f:
    f.write('tensorflow\npandas\nnumpy\nscikit-learn\ngradio\n')

# 2. Membuat file app.py menggunakan Gradio (Otomatis menyediakan API)
api_code = """
import gradio as gr
import tensorflow as tf
import pickle
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load Model dan Aset
model = tf.keras.models.load_model('transaction_classifier_model.keras')
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)
with open('label_encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)

def predict_api(description):
    # Preprocessing
    seq = tokenizer.texts_to_sequences([description])
    padded = pad_sequences(seq, maxlen=20, padding='post')

    # Prediksi
    prediction = model.predict(padded, verbose=0)
    class_idx = np.argmax(prediction, axis=1)[0]
    category = encoder.inverse_transform([class_idx])[0]
    confidence = float(np.max(prediction))

    return {"category": category, "confidence": confidence}

# Interface Gradio
demo = gr.Interface(
    fn=predict_api,
    inputs=gr.Textbox(label="Deskripsi Transaksi"),
    outputs=gr.JSON(label="Hasil Klasifikasi"),
    title="Transaction Classifier API"
)

if __name__ == "__main__":
    demo.launch()
"""

with open('app.py', 'w') as f:
    f.write(textwrap.dedent(api_code))

print("✅ File 'app.py' (Gradio) dan 'requirements.txt' telah diperbarui.")

✅ File 'app.py' (Gradio) dan 'requirements.txt' telah diperbarui.


In [21]:
from google.colab import files
# Download file pendukung untuk API
files.download('app.py')
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>